# Moment scaling

This notebook performs the moment scaling analysis of individual seismic acceleration signals, following the framework of Vollmer et al. (2024). Results from the PDF 
analysis (notebook 03a) are imported at the end to build the complete summary table.

## 1. Imports and visualization settings

In [ ]:
from pathlib import Path
import pandas as pd
import logging
from src import (
    integrate_to_velocity,
    integrate_to_displacement,
    set_plot_style,
    plot_onset_diagnostic,
    plot_onset_distribution,
)
colors = set_plot_style()
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger()
def check(condition, message):
    if condition:
        logger.info(message)
    else:
        raise ValueError(message)
logger.info("Environment ready")

INFO | Environment ready


## 2. Data loading

In [3]:
logger.info("Loading preprocessed data...")
df_acc_clean = pd.read_parquet('../data/processed/02_signals/acc_preprocessed_pdf.parquet')
df_acc_long  = pd.read_parquet('../data/processed/02_signals/acc_preprocessed_scaling.parquet')
check(len(df_acc_clean) > 0, f"All signals loaded: {df_acc_clean['file'].nunique()} files")
check(len(df_acc_long) > 0,  f"Long signals loaded: {df_acc_long['file'].nunique()} files")

FIGURES_DIR = Path('../figures/03_single_signal/03b_moment_scaling')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
(FIGURES_DIR / 'scaling' / 'displacement' / 'event_window').mkdir(parents=True, exist_ok=True)
check(FIGURES_DIR.exists(), f"Figures directory ready: {FIGURES_DIR}")

INFO | Loading preprocessed data...
INFO | All signals loaded: 66 files
INFO | Long signals loaded: 48 files
INFO | Figures directory ready: ../figures/03_single_signal/03b_moment_scaling


## 3. Moment scaling analysis

The scaling behavior of statistical moments is investigated to detect
signatures of anomalous or strongly anomalous diffusion, following the
framework of Vollmer et al. (2024).

For a process $x(t)$, the displacement over a time scale $\tau$ starting
from $t_0$ is defined as:

$$\Delta x(\tau, t_0) = x(t_0 + \tau) - x(t_0)$$

The $q$-th order moment is estimated as a temporal average over all
possible starting points $t_0$ (sliding window):

$$M_q(\tau) = \langle |\Delta x(\tau, t_0)|^q \rangle_{t_0}
= \frac{1}{N - \tau} \sum_{t_0=0}^{N-\tau-1} |\Delta x(\tau, t_0)|^q$$

If the process exhibits scaling, the moments obey a power law in $\tau$:

$$M_q(\tau) \sim \tau^{\zeta(q)}$$

For normal diffusion, $\zeta(q) = q/2$ (linear in $q$). Strong anomalous
diffusion is characterized by a piecewise-linear spectrum with a crossover
at $q = \alpha$ (Vollmer et al., 2024, Eq.~1b):

$$\zeta(q) = \begin{cases} \xi\, q & q \leq \alpha \\ \zeta_0\, q - \alpha(\zeta_0 - \xi) & q > \alpha \end{cases}$$

Three definitions of the process $x(t)$ are explored and compared:

- **Section 4** — $x(t) = a(t)$: the acceleration signal itself; increments
  are differences of acceleration values $\Delta a(\tau, t_0) = a(t_0+\tau) - a(t_0)$.
- **Section 5** — $x(t) = v(t) = \sum_{k} a(k)\,\Delta t$: the ground velocity,
  obtained by integrating the acceleration once.
- **Section 6** — $x(t) = \int v(t)\,dt$: the ground displacement,
  obtained by integrating the acceleration twice.

All three versions use a sampling interval $\Delta t = 0.005$\,s (200\,Hz).
The 6 near-field stations with short recordings (SURF, BRZ, BHB, CRI, SLZ, SAV)
are excluded from all moment scaling analyses, leaving 48 signals.

### 3.1 Acceleration

#### 3.1.1 Computation of q-th order moments

The process is defined as $x(t) = a(t)$, the normalized acceleration signal.
Increments are computed as:

$$\Delta a(\tau, t_0) = a(t_0 + \tau) - a(t_0)$$

Moments $M_q(\tau)$ are computed for moment orders $q \in \{0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0\}$ and time scales $\tau \in \{10, 50, 100, 200, 500, 1000, 2000, 5000, 10000\}$ samples.

INFO | Computing acceleration increments — full signal...
INFO | Increments computed: (40358880, 6)
INFO | Increments saved: (40358880, 6)
INFO | Computing acceleration moments — full signal...
INFO | Moments computed: (16416, 6)
INFO | Moments saved: (16416, 6)


#### 3.1.2 Scaling exponents estimation

For each signal and each moment order $q$, the scaling exponent $\zeta(q)$ 
is estimated by linear regression of $\log M_q(\tau)$ vs $\log \tau$:

$$\log M_q(\tau) = \zeta(q) \log \tau + c$$

The slope $\zeta(q)$ is the scaling exponent. A plot of $\zeta(q)$ vs $q$ 
is produced for each signal, with the reference line $\zeta(q) = q/2$ 
corresponding to normal diffusion.

INFO | Computing scaling exponents — acceleration, full signal...
INFO | Exponents saved.


Saved: 48/48 individual plots
All individual plots saved successfully!


#### 3.1.3 Linearity check

The linearity of $\zeta(q)$ vs $q$ is assessed by comparing two models 
using AIC:

- **Linear**: $\zeta(q) = aq + b$
- **Quadratic**: $\zeta(q) = aq^2 + bq + c$

If the quadratic model is preferred, the scaling is non-linear, 
indicating anomalous diffusion.

A piecewise linear fit is also performed to detect the presence of 
two distinct scaling regimes, consistent with the framework of strong 
anomalous diffusion (Vollmer et al., 2021):

$$\zeta(q) = \begin{cases} \phi q & q \leq q^* \\ \lambda q - q^*(\lambda - \phi) & q > q^* \end{cases}$$

where $q^*$ is the breakpoint and $\phi$, $\lambda$ are the slopes of 
the two regimes. The breakpoint $q^*$ is estimated by minimizing the 
total residual sum of squares.

INFO | Linearity test and piecewise fit — acceleration, full signal...
INFO | Linearity saved.


Saved: 48/48 individual plots
All individual plots saved successfully!

Best fit by AIC:
best_fit
quadratic    46
linear        2
Name: count, dtype: int64


INFO | Piecewise saved.


Saved: 48/48 individual plots
All individual plots saved successfully!


### 3.2 Velocity

#### 3.2.1 Computation of q-th order moments

The process is defined as the ground velocity, obtained by integrating
the acceleration once:

$$v(t) = \sum_{k=0}^{t} a(k)\,\Delta t$$

Increments are computed as:

$$\Delta v(\tau, t_0) = v(t_0 + \tau) - v(t_0)$$

Moments $M_q(\tau)$ are computed for the same $q$ and $\tau$ values as
in Section 3.1.

INFO | Integrating acceleration to velocity — full signal...
INFO | Velocity range: [-0.0458, 0.0361] cm/s
INFO | Computing velocity increments — full signal...
INFO | Increments computed: (40358880, 6)
INFO | Increments saved: (40358880, 6)
INFO | Computing velocity moments — full signal...
INFO | Moments computed: (16416, 6)
INFO | Moments saved: (16416, 6)


#### 3.2.2 Scaling exponents estimation

For each signal and each moment order $q$, the scaling exponent $\zeta(q)$ 
is estimated by linear regression of $\log M_q(\tau)$ vs $\log \tau$:

$$\log M_q(\tau) = \zeta(q) \log \tau + c$$

The slope $\zeta(q)$ is the scaling exponent. A plot of $\zeta(q)$ vs $q$ 
is produced for each signal, with the reference line $\zeta(q) = q/2$ 
corresponding to normal diffusion.

INFO | Computing scaling exponents — velocity, full signal...
INFO | Exponents saved.


Saved: 48/48 individual plots
All individual plots saved successfully!


#### 3.2.3 Linearity check

The linearity of $\zeta(q)$ vs $q$ is assessed by comparing two models 
using AIC:

- **Linear**: $\zeta(q) = aq + b$
- **Quadratic**: $\zeta(q) = aq^2 + bq + c$

If the quadratic model is preferred, the scaling is non-linear, 
indicating anomalous diffusion.

A piecewise linear fit is also performed to detect the presence of 
two distinct scaling regimes, consistent with the framework of strong 
anomalous diffusion (Vollmer et al., 2021):

$$\zeta(q) = \begin{cases} \phi q & q \leq q^* \\ \lambda q - q^*(\lambda - \phi) & q > q^* \end{cases}$$

where $q^*$ is the breakpoint and $\phi$, $\lambda$ are the slopes of 
the two regimes. The breakpoint $q^*$ is estimated by minimizing the 
total residual sum of squares.

INFO | Linearity test and piecewise fit — velocity, full signal...
INFO | Linearity saved.


Saved: 48/48 individual plots
All individual plots saved successfully!

Best fit by AIC:
best_fit
quadratic    47
linear        1
Name: count, dtype: int64


INFO | Piecewise saved.


Saved: 48/48 individual plots
All individual plots saved successfully!


### 3.3 Displacement

#### 3.3.1 Computation of q-th order moments

The process is defined as the ground displacement, obtained by integrating
the acceleration twice:

$$v(t) = \sum_{k=0}^{t} a(k)\,\Delta t, \qquad
x(t) = \sum_{k=0}^{t} v(k)\,\Delta t$$

Increments are computed as:

$$\Delta x(\tau, t_0) = x(t_0 + \tau) - x(t_0)$$

Moments $M_q(\tau)$ are computed for the same $q$ and $\tau$ values as
in Sections 3.1 and 3.2.

INFO | Integrating acceleration to displacement — full signal...
INFO | Displacement range: [-0.002582, 0.004235] cm
INFO | Computing displacement increments — full signal...
INFO | Increments computed: (40358880, 6)
INFO | Increments saved: (40358880, 6)
INFO | Computing displacement moments — full signal...
INFO | Moments computed: (16416, 6)
INFO | Moments saved: (16416, 6)


#### 3.3.2 Scaling exponents estimation

For each signal and each moment order $q$, the scaling exponent $\zeta(q)$ 
is estimated by linear regression of $\log M_q(\tau)$ vs $\log \tau$:

$$\log M_q(\tau) = \zeta(q) \log \tau + c$$

The slope $\zeta(q)$ is the scaling exponent. A plot of $\zeta(q)$ vs $q$ 
is produced for each signal, with the reference line $\zeta(q) = q/2$ 
corresponding to normal diffusion.

INFO | Computing scaling exponents — displacement, full signal...
INFO | Exponents saved.


Saved: 48/48 individual plots
All individual plots saved successfully!


#### 3.3.3 Linearity check

The linearity of $\zeta(q)$ vs $q$ is assessed by comparing two models 
using AIC:

- **Linear**: $\zeta(q) = aq + b$
- **Quadratic**: $\zeta(q) = aq^2 + bq + c$

If the quadratic model is preferred, the scaling is non-linear, 
indicating anomalous diffusion.

A piecewise linear fit is also performed to detect the presence of 
two distinct scaling regimes, consistent with the framework of strong 
anomalous diffusion (Vollmer et al., 2021):

$$\zeta(q) = \begin{cases} \phi q & q \leq q^* \\ \lambda q - q^*(\lambda - \phi) & q > q^* \end{cases}$$

where $q^*$ is the breakpoint and $\phi$, $\lambda$ are the slopes of 
the two regimes. The breakpoint $q^*$ is estimated by minimizing the 
total residual sum of squares.

INFO | Linearity test and piecewise fit — displacement, full signal...
INFO | Linearity saved.


Saved: 48/48 individual plots
All individual plots saved successfully!

Best fit by AIC:
best_fit
quadratic    48
Name: count, dtype: int64


INFO | Piecewise saved.


Saved: 48/48 individual plots
All individual plots saved successfully!


#### Increment distributions and tail exponents

The empirical CCDF of $|\Delta x(\tau)|$ is plotted in log-log scale for
six representative time scales $\tau \in \{10, 100, 500, 1000, 3000, 5000\}$
samples. The tail exponent is estimated independently via the Hill estimator
and via a linear fit on the log-log CCDF (top 10% of values).

## 4. Summary

A summary table collects the main results from all analyses for each signal.
For the moment scaling columns, results from the displacement version (Section 3.3)
are reported as the physically most meaningful choice.

| Column | Description |
|--------|-------------|
| `kurtosis` | Excess kurtosis $\kappa$ |
| `skewness` | Skewness $\gamma$ |
| `non_gaussian` | Anderson-Darling test result ($\alpha = 0.05$) |
| `best_fit_aic` | Best fitting distribution by AIC |
| `student_t_df` | Student-$t$ degrees of freedom $\nu$ |
| `power_law_exp` | Power law exponent $\hat{\alpha}$ (Hill estimator) |
| `q_break` | Piecewise scaling breakpoint $q^*$ |
| `slope_low` | Scaling slope for $q \leq q^*$ |
| `slope_high` | Scaling slope for $q > q^*$ |
| `beta` | Autocorrelation scaling exponent $\beta$ |


### 4.1 Acceleration

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/03a_pdf_analysis/pdf_summary.parquet'

### 4.2 Velocity

### 4.3 Displacement